Write a query that'll identify returning active users. A returning active user is a user that has made a second purchase within 7 days of any other of their purchases. Output a list of user_ids of these returning active users.


In [0]:
%skip
CREATE TABLE ska_catalog.bronze.amazon_transactions(id int, user_id int, item varchar(15), created_at TIMESTAMP, revenue int);

INSERT INTO ska_catalog.bronze.amazon_transactions VALUES (1,109,'milk','2020-03-03 00:00:00',123),(2,139,'biscuit','2020-03-18 00:00:00', 421), (3,120,'milk','2020-03-18 00:00:00',176), (4,108,'banana','2020-03-18 00:00:00',862), (5,130,'milk','2020-03-28 00:00:00',333), (6,103,'bread','2020-03-29 00:00:00',862), (7,122,'banana','2020-03-07 00:00:00',952), (8,125,'bread','2020-03-13 00:00:00',317), (9,139,'bread','2020-03-30 00:00:00',929), (10,141,'banana','2020-03-17 00:00:00',812), (11,116,'bread','2020-03-31 00:00:00',226), (12,128,'bread','2020-03-04 00:00:00',112), (13,146,'biscuit','2020-03-04 00:00:00',362), (14,119,'banana','2020-03-28 00:00:00',127), (15,142,'bread','2020-03-09 00:00:00',503), (16,122,'bread','2020-03-06 00:00:00',593), (17,128,'biscuit','2020-03-24 00:00:00',160), (18,112,'banana','2020-03-24 00:00:00',262), (19,149,'banana','2020-03-29 00:00:00',382), (20,100,'banana','2020-03-18 00:00:00',599);

In [0]:
SELECT * FROM ska_catalog.bronze.amazon_transactions

In [0]:
SELECT DISTINCT a.user_id FROM ska_catalog.bronze.amazon_transactions a
JOIN ska_catalog.bronze.amazon_transactions b
WHERE a.user_id = b.user_id AND a.created_at < b.created_at AND DATEDIFF(day,a.created_at,  b.created_at) < 7

In [0]:
-- What is the total revenue generated from each item?
SELECT
  initcap(item) AS `ITEM`,
  SUM(revenue) AS `TOTAL_REVENUE`
FROM ska_catalog.bronze.amazon_transactions
GROUP BY ITEM
ORDER BY TOTAL_REVENUE DESC;

In [0]:
-- Which item had the highest revenue?
SELECT ITEM AS `High revenue item` FROM (
  SELECT
  initcap(item) AS `ITEM`,
  SUM(revenue) AS `TOTAL_REVENUE`
FROM ska_catalog.bronze.amazon_transactions
GROUP BY ITEM
ORDER BY TOTAL_REVENUE DESC
)
LIMIT 1;

In [0]:
-- Which item had the highest number of transactions?

SELECT item AS `ITEM`, COUNT(*) `No_of_transactions`
FROM ska_catalog.bronze.amazon_transactions
GROUP BY ITEM
ORDER BY No_of_transactions DESC;

In [0]:
-- What is the average revenue per transaction for each item?
SELECT
  item AS `ITEM`,
  ROUND(Avg(revenue),2) AS `AVG_REVENUE`
FROM ska_catalog.bronze.amazon_transactions
GROUP BY ITEM
ORDER BY AVG_REVENUE DESC;

In [0]:
-- List all transactions made by user_id 139.
SELECT * FROM ska_catalog.bronze.amazon_transactions WHERE user_id = 139;

In [0]:
-- Which day had the third highest total revenue?
SELECT date_format(created_at, 'yyyy-MM-dd') AS `DATE`, sum(revenue) AS `TOTAL_REVENUE` FROM ska_catalog.bronze.amazon_transactions
GROUP BY created_at
ORDER BY TOTAL_REVENUE DESC
LIMIT 1 OFFSET 2;

In [0]:
-- How many unique users purchased 'banana'?
SELECT COUNT(DISTINCT user_id) AS unique_users_banana
FROM ska_catalog.bronze.amazon_transactions
WHERE item = 'banana';

In [0]:
-- What is the total revenue generated in the month of March 2020?
SELECT SUM(revenue) AS total_revenue_march_2020
FROM ska_catalog.bronze.amazon_transactions
WHERE date_format(created_at,'yyyy-MM-dd') >= '2020-03-01' AND date_format(created_at,'yyyy-MM-dd') < '2020-04-01';

In [0]:
-- Which users made more than one transaction?
SELECT user_id
FROM ska_catalog.bronze.amazon_transactions
GROUP BY user_id
HAVING COUNT(*) > 1;

In [0]:
-- What is the revenue trend for 'bread' over time?
SELECT
  date_format(created_at, 'yyyy-MM-dd') AS `DATE`,
  SUM(revenue) AS `BREAD_REVENUE`
FROM ska_catalog.bronze.amazon_transactions
WHERE item = 'bread'
GROUP BY DATE
ORDER BY DATE ASC;

In [0]:
-- Rank the items based on total revenue in descending order.
SELECT
  item AS `ITEM`,
  SUM(revenue) AS `TOTAL_REVENUE`,
  RANK() OVER (ORDER BY SUM(revenue) DESC) AS `REVENUE_RANK`
FROM ska_catalog.bronze.amazon_transactions
GROUP BY item
ORDER BY REVENUE_RANK;